# 🦈 Matrix Compression with SVD

📷 Photo ➡️ 🔢 Numbers ➡️ 🍰 Layers ➡️ ✂️ Keep the top **k** ➡️ 🖼️ Smaller photo

## 📦 Libraries and Tools

In [ ]:
# 🔢 Numbers and tables (matrices)
import numpy as np

# 🖼️ Open photos
from PIL import Image

# 📊 Draw pictures and charts
import matplotlib.pyplot as plt

# 📁 Where to save the figures for the slides
from pathlib import Path
OUT_DIR = Path("image")
OUT_DIR.mkdir(exist_ok=True)

print("✅ Tools loaded successfully!")

## 🔢 Step 1 — A photo is a table of numbers
Each pixel is a number: **0 = black ⚫ … 255 = white ⚪**. A table of numbers = a **matrix**.

In [ ]:
# 🦈 Open the photo as numbers
img = np.asarray(Image.open("images/shark.jpg").convert("RGB"), dtype=float)
gray = img.mean(axis=2)  # ⚫⚪ brightness only

# 🔍 Zoom into an 8 x 8 square
top, left = 300, 520
patch = gray[top:top + 8, left:left + 8]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.5, 1]})
ax1.imshow(img.astype(np.uint8))
ax1.add_patch(plt.Rectangle((left, top), 8, 8, fill=False, color="red", lw=3))
ax1.annotate("zoom", (left + 8, top), xytext=(left + 120, top - 120),
             color="red", fontsize=15, arrowprops={"color": "red", "lw": 2})
ax1.set_title("The photo", fontsize=16)
ax1.axis("off")

ax2.imshow(patch, cmap="gray", vmin=0, vmax=255)
for r in range(8):
    for c in range(8):
        v = int(patch[r, c])
        ax2.text(c, r, v, ha="center", va="center", color="white" if v < 128 else "black")
ax2.set_title("Zoomed in: numbers", fontsize=16)
ax2.axis("off")

fig.savefig(OUT_DIR / "00_photo_is_numbers.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ m x n =", img.shape[0], "x", img.shape[1], "pixels, 3 colours (R, G, B)")

## 🍰 Step 2 — SVD splits the photo into layers
$A = \sigma_1 u_1 v_1^T + \sigma_2 u_2 v_2^T + \cdots$ — most important layer first ⭐
(shown in black & white to keep it simple)

In [ ]:
# 🍰 SVD: U, S (importance of each layer), Vt
U, S, Vt = np.linalg.svd(gray, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, k in zip(axes, [1, 3, 10, 50]):
    # ➕ add up the first k layers
    A_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    ax.imshow(np.clip(A_k, 0, 255), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"k = {k}", fontsize=15)
    ax.axis("off")

fig.savefig(OUT_DIR / "01_svd_layers.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Importance of layers 1, 2, 3:", S[:3].round(0), "... layer 200:", S[199].round(0))

## ✂️ Step 3 — Keep only the first k layers
🎛️ **k** = how many layers we keep.

💾 Storage: $k(m+n+1)$ numbers instead of $m \times n$ · 🎯 Quality: PSNR in dB (30+ = looks very good)

In [ ]:
# 🍰 Split each colour (R, G, B) into layers
def load(path):
    img = np.asarray(Image.open(path).convert("RGB"), dtype=float)
    return img, [np.linalg.svd(img[:, :, c], full_matrices=False) for c in range(3)]


# ✂️ Rebuild the photo from the first k layers
def compress(svds, k):
    channels = [U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :] for U, S, Vt in svds]
    return np.clip(np.stack(channels, axis=2), 0, 255).astype(np.uint8)


# 💾 Space used compared with the original
def storage(img, k):
    m, n, _ = img.shape
    return k * (m + n + 1) / (m * n)


# 🎯 Quality score (dB)
def psnr(img, approx):
    return 10 * np.log10(255 ** 2 / np.mean((img - approx) ** 2))


data = {name: load(f"images/{name.lower()}.jpg") for name in ["Shark", "Deer", "Leopard"]}
print("✅ Ready!")

## 🖼️ Step 4 — Results
Each picture: **k** · 💾 storage used · 🎯 quality (dB). Small k = blurry, big k = sharp.

In [ ]:
for name, (img, svds) in data.items():
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    axes[0].imshow(img.astype(np.uint8))
    axes[0].set_title("Original · 100%", fontsize=14)

    # ✂️ More layers = sharper photo, bigger file
    for ax, k in zip(axes[1:], [5, 20, 200]):
        approx = compress(svds, k)
        ax.imshow(approx)
        ax.set_title(f"k = {k} · {storage(img, k):.0%} · {psnr(img, approx):.0f} dB", fontsize=14)

    for ax in axes:
        ax.axis("off")
    fig.savefig(OUT_DIR / f"02_results_{name.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 📈 Step 5 — How many layers are worth it?
⬆️ Higher = sharper · ⬅️ further left = smaller file · dashed line = 30 dB (looks very good 😍)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for (name, (img, svds)), color in zip(data.items(), ["#2a78d6", "#eb6834", "#1baf7a"]):
    ks = list(range(1, 20)) + list(range(20, 401, 10))
    ax.plot([storage(img, k) * 100 for k in ks], [psnr(img, compress(svds, k)) for k in ks],
            color=color, lw=2.5, label=name)

# 😍 30 dB = looks very good
ax.axhline(30, color="gray", ls="--")
ax.text(1, 30.5, "30 dB = looks very good", color="gray")

ax.set(xlim=(0, 100), xlabel="storage used (%)", ylabel="quality (dB)",
       title="Quality vs. storage")
ax.legend(loc="lower right")
fig.savefig(OUT_DIR / "03_quality_vs_storage.png", dpi=150, bbox_inches="tight")
plt.show()

## 🏁 Summary

1. 🔢 A photo is a matrix.

2. 🍰 SVD splits it into layers, most important first.

3. ✂️ Keep k layers → small k = small & blurry, big k = sharp & bigger.

4. 🦈 5% of the data is enough to see the shark; ~45% looks very good (30 dB).